<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/InstutionalFiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
pip install requests pandas tabulate

In [50]:
import requests
import pandas as pd


class InstitutionalEngine:

    BASE_URL = "https://data.businessquant.com/13f"

    def __init__(self, api_key):
        self.api_key = api_key

    def _request(self, mode, ticker):

        url = (
            f"{self.BASE_URL}"
            f"?mode={mode}"
            f"&ticker_issuer={ticker}"
            f"&api_key={self.api_key}"
        )

        r = requests.get(url)

        if r.status_code != 200:
            raise Exception(r.text)

        return r.json()

    def get_summary(self, ticker):

        return self._request("summary", ticker)

    def get_stats(self, ticker):

        return self._request("stats", ticker)

    def get_topholders(self, ticker):

        return self._request("topholders", ticker)

    def build_report(self, ticker):
      # Get summary data - we'll take the most recent quarter
      summary_response = self.get_summary(ticker)
      summaries = summary_response.get("data", [])
      if not summaries:
            raise Exception(f"No data found for ticker {ticker}")

      # Sort by quarter (most recent first) - handles '2025-12-31T00:00:00' format
      latest_summary = sorted(
            summaries,
            key=lambda x: x.get("quarter", ""),
            reverse=True
        )[0]
      # Get top holders
      holders_response = self.get_topholders(ticker)
      holders_data = holders_response.get("data", [])
      #Filter holders to match the latest quarter (if multiple quarters returned)
      latest_quarter = latest_summary["quarter"]
      holders_data = [h for h in holders_data if h.get("quarter") == latest_quarter]
      holders = pd.DataFrame(holders_data)
      # === Build Report ===
      report = {}

      report["Ticker"] = latest_summary.get("ticker")
      report["Quarter"] = latest_summary.get("quarter")
      report["Institution Count"] = latest_summary.get("institutions_total_count")
      report["Bought Count"] = latest_summary.get("institutions_bought_count")
      report["Sold Count"] = latest_summary.get("institutions_sold_count")
      report["Held Count"] = latest_summary.get("institutions_held_count")
      report["Total Shares"] = latest_summary.get("institutions_total_shares")
      report["Ownership %"] = latest_summary.get("institutions_shares_pct_outstanding")
      report["QoQ Shares Change"] = latest_summary.get("shares_changed_qoq")
      report["QoQ %"] = latest_summary.get("shares_changed_qoq_pct")
      report["YoY Shares Change"] = latest_summary.get("shares_changed_yoy")

      report["Top10 Concentration"] = (
            holders["institution_pct"].head(10).sum()
            if not holders.empty else 0
        )

      report["Net Buying"] = (
            latest_summary.get("institutions_bought_shares", 0) -
            latest_summary.get("institutions_sold_shares", 0)
        )

      report["Buy/Sell Ratio"] = (
            latest_summary.get("institutions_bought_count", 0) /
            max(latest_summary.get("institutions_sold_count", 0), 1)
        )

      # Top Buyers & Sellers
      if not holders.empty:
        buyers = holders.sort_values("shares_change_qoq", ascending=False)
        sellers = holders.sort_values("shares_change_qoq")
        report["Top Buyers"] = buyers[[
                "name_filer_short", "shares_change_qoq", "shares_change_qoq_pct"
            ]].head(10).reset_index(drop=True)

        report["Top Sellers"] = sellers[[
                "name_filer_short", "shares_change_qoq", "shares_change_qoq_pct"
            ]].head(10).reset_index(drop=True)
      else:
        report["Top Buyers"] = pd.DataFrame()
        report["Top Sellers"] = pd.DataFrame()

      return report



# instutional score
def institutional_score(report):

    score = 0

    # Holder Sentiment
    if report["Buy/Sell Ratio"] > 1.5:
        score += 20
    elif report["Buy/Sell Ratio"] > 1.2:
        score += 15
    elif report["Buy/Sell Ratio"] > 1:
        score += 10

    # QoQ Ownership
    if report["QoQ %"] > 5:
        score += 20
    elif report["QoQ %"] > 2:
        score += 15
    elif report["QoQ %"] > 0:
        score += 10

    # Net Buying

    if report["Net Buying"] > 0:
        score += 20

    # Ownership

    if report["Ownership %"] > 70:
        score += 15
    elif report["Ownership %"] > 50:
        score += 10

    # Concentration

    if report["Top10 Concentration"] < 20:
        score += 15
    elif report["Top10 Concentration"] < 40:
        score += 10
    else:
        score += 5

    # Buyers vs Sellers

    if report["Bought Count"] > report["Sold Count"]:
        score += 10

    return score

In [49]:
API_KEY =  "9cc8c875a6c2b773eef673e93ced70d7"

#engine = InstitutionalEngine(API_KEY)

#report = engine.build_report("MZTI")

#print(report)
#df = pd.DataFrame([report])
# Drop unwanted columns
#df = df.drop(columns=['Top Buyers', 'Top Sellers'], errors='ignore')

#df['Institutional Score'] = df.apply(lambda row: institutional_score(row.to_dict()), axis=1)


#print()

#df

,Ticker,Quarter,Institution Count,Bought Count,Sold Count,Held Count,Total Shares,Ownership %,QoQ Shares Change,QoQ %,YoY Shares Change,Top10 Concentration,Net Buying,Buy/Sell Ratio,Institutional Score
0,MZTI,2025-12-31T00:00:00,332,154,146,32,23556907.0,85.927073,6748215.0,40.147175,5964471.0,0,6748215.0,1.054795,90


In [51]:
def build_institutional_dataframe(tickers, api_key, max_retries=2):
    """
    Process multiple tickers and return a single DataFrame with scores.
    """
    engine = InstitutionalEngine(api_key)
    reports = []

    for ticker in tickers:
        try:
            print(f"Processing {ticker}...", end=" ")

            report = engine.build_report(ticker)
            score = institutional_score(report)

            # Create a flat row for the DataFrame (drop complex columns)
            row = {
                "Ticker": report["Ticker"],
                "Quarter": report["Quarter"],
                "Institution Count": report["Institution Count"],
                "Bought Count": report["Bought Count"],
                "Sold Count": report["Sold Count"],
                "Held Count": report["Held Count"],
                "Ownership %": report["Ownership %"],
                "QoQ %": report["QoQ %"],
                "Buy/Sell Ratio": report["Buy/Sell Ratio"],
                "Net Buying": report["Net Buying"],
                "Top10 Concentration": report["Top10 Concentration"],
                "Institutional Score": score
            }

            reports.append(row)
            print("✓")

        except Exception as e:
            print(f"✗ Error: {e}")
            continue  # Skip failed tickers

    if not reports:
        raise Exception("No data retrieved for any ticker")

    # Create final DataFrame
    df = pd.DataFrame(reports)

    # Optional: Sort by score (highest first)
    df = df.sort_values(by="Institutional Score", ascending=False).reset_index(drop=True)

    # Drop any unwanted columns (you can modify this list)
    columns_to_drop = ['Top Buyers', 'Top Sellers']  # Add more if needed
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')

    return df


# ============================
# HOW TO USE IT
# ============================

tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]   # Add your list here

api_key = "9cc8c875a6c2b773eef673e93ced70d7"

df = build_institutional_dataframe(tickers, api_key)

# Display results
print(f"\n✅ Processed {len(df)} tickers successfully!")
display(df)                    # Nice display in Jupyter
# df.to_csv("institutional_scores.csv", index=False)   # Uncomment to save

Processing AAPL... ✗ Error: {"detail":"Rate limit exceeded. Limit: 30 req/day"}
Processing MSFT... ✗ Error: {"detail":"Rate limit exceeded. Limit: 30 req/day"}
Processing GOOGL... ✗ Error: {"detail":"Rate limit exceeded. Limit: 30 req/day"}
Processing AMZN... ✗ Error: {"detail":"Rate limit exceeded. Limit: 30 req/day"}
Processing NVDA... ✗ Error: {"detail":"Rate limit exceeded. Limit: 30 req/day"}


Exception: No data retrieved for any ticker

In [ ]:
def build_reports(self, ticker):
        # Get summary data - we'll take the most recent quarter
        summary = self.get_summary(ticker)["data"][0]

        holders = pd.DataFrame(
            self.get_topholders(ticker)["data"]
        )

        report = {}

        report["Ticker"] = summary["ticker"]

        report["Quarter"] = summary["quarter"]

        report["Institution Count"] = summary["institutions_total_count"]

        report["Bought Count"] = summary["institutions_bought_count"]

        report["Sold Count"] = summary["institutions_sold_count"]

        report["Held Count"] = summary["institutions_held_count"]

        report["Total Shares"] = summary["institutions_total_shares"]

        report["Ownership %"] = summary[
            "institutions_shares_pct_outstanding"
        ]

        report["QoQ Shares Change"] = summary[
            "shares_changed_qoq"
        ]

        report["QoQ %"] = summary[
            "shares_changed_qoq_pct"
        ]

        report["YoY Shares Change"] = summary[
            "shares_changed_yoy"
        ]

        report["Top10 Concentration"] = (
            holders["institution_pct"]
            .head(10)
            .sum()
        )

        report["Net Buying"] = (
            summary["institutions_bought_shares"]
            -
            summary["institutions_sold_shares"]
        )

        report["Buy/Sell Ratio"] = (
            summary["institutions_bought_count"]
            /
            max(summary["institutions_sold_count"],1)
        )

        buyers = holders.sort_values(
            "shares_change_qoq",
            ascending=False
        )

        sellers = holders.sort_values(
            "shares_change_qoq"
        )

        report["Top Buyers"] = buyers[
            [
                "name_filer_short",
                "shares_change_qoq",
                "shares_change_qoq_pct"
            ]
        ].head(10)

        report["Top Sellers"] = sellers[
            [
                "name_filer_short",
                "shares_change_qoq",
                "shares_change_qoq_pct"
            ]
        ].head(10)

        return report